# LangChain: Documentation RAG Agent

This notebook recreates the documentation RAG agent from the PydanticAI lesson using **LangChain**.

LangChain provides a large ecosystem of integrations and a flexible agent architecture. We use the same `search_tools` and `AGENT_INSTRUCTIONS` modules from the earlier lessons.

## Install dependencies

```bash
uv add langchain langchain-openai
```

`langchain` provides the core agent logic, while `langchain-openai` adds OpenAI model support. LangChain splits model integrations into separate packages so you install only what you need.

### Short production reminder

For production, replace print-based callbacks with structured logs or tracing events. Include request/run IDs, tool name, latency, status, and redacted argument summaries. Do not log sensitive content or full tool arguments by default. Keep callbacks observational; enforce authorization and validation inside the tool execution boundary.


In [1]:
!uv add langchain langchain-openai

Resolved 166 packages in 8ms
Checked 162 packages in 113ms


In [2]:
from search_tools import create_search_tools
from common import AGENT_INSTRUCTIONS

search_tools = create_search_tools()

ImportError: cannot import name 'chunkdocuments' from 'gitsource' (/workspaces/ai-engineering-buildcamp/.venv/lib/python3.14/site-packages/gitsource/__init__.py)

## Create the model

`ChatOpenAI` is LangChain's client for OpenAI chat models. By default it uses the Chat Completions API.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

## Define tools

LangChain can wrap functions with the `@tool` decorator. The function docstring becomes the tool description sent to the model.

However, because our `SearchTools` class already has documented methods, we can avoid redefining the functions by using `StructuredTool.from_function()`. It reads the function signature and docstring directly.

In [ ]:
from langchain_core.tools import StructuredTool

tools = [
    StructuredTool.from_function(search_tools.search),
    StructuredTool.from_function(search_tools.get_file),
]

tools

## Create the agent

`create_agent` wires together the model, tools, and system prompt.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=AGENT_INSTRUCTIONS,
)

## Run the agent

LangChain agents use a message-based interface. We pass a dictionary containing a `messages` list.

The result is also a dictionary. Its final assistant message is the last item in `result["messages"]`.

In [ ]:
query = "how do I use evidently to monitor my machine learning models?"

result = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

print(result["messages"][-1].content)

## Inspect the message history

LangChain uses typed message classes:

- `HumanMessage`: user input
- `AIMessage`: model output; may include text and tool calls
- `ToolMessage`: result returned by a tool

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def print_langchain_messages(messages):
    for msg in messages:
        if isinstance(msg, HumanMessage):
            print(f"User: {msg.content[:200]}")

        elif isinstance(msg, AIMessage):
            if msg.content:
                print(f"Assistant: {msg.content[:300]}")

            if msg.tool_calls:
                for tool_call in msg.tool_calls:
                    print(
                        f"Tool call: {tool_call['name']}({tool_call['args']})"
                    )

        elif isinstance(msg, ToolMessage):
            print(f"Tool result: {str(msg.content)[:200]}...")

        print()


print_langchain_messages(result["messages"])

## Follow-up conversation

To continue a conversation, pass the previous messages plus the new user message.

LangChain does not automatically retain history between separate `invoke()` calls unless you use a memory/checkpointer or a conversation-management abstraction.

In [ ]:
follow_up = "show me the code"

previous_messages = result["messages"]

new_input = previous_messages + [
    {"role": "user", "content": follow_up}
]

result2 = agent.invoke({"messages": new_input})

print(result2["messages"][-1].content)

In [ ]:
# Show only messages produced during the follow-up run
follow_up_messages = result2["messages"][len(previous_messages):]

print_langchain_messages(follow_up_messages)

## Use the Responses API

By default, `ChatOpenAI` uses the Chat Completions API. To use OpenAI's Responses API instead, set `use_responses_api=True`.

In [ ]:
llm_responses = ChatOpenAI(
    model="gpt-4o-mini",
    use_responses_api=True,
)

agent_responses = create_agent(
    model=llm_responses,
    tools=tools,
    system_prompt=AGENT_INSTRUCTIONS,
)

result_responses = agent_responses.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

print(result_responses["messages"][-1].content)

## Production-style observability

In production, do not rely on `print()` for agent telemetry. Use structured logging or tracing.

The callback below records tool-call events without executing tools or making authorization decisions. Authorization and validation belong inside the tool function or an outer service boundary.

In [ ]:
import time
import logging
import uuid

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
)

logger = logging.getLogger("langchain_docs_agent")


def summarize_args(args, limit: int = 120) -> str:
    """Return a short, redacted argument summary for logs."""
    text = str(args)
    return text if len(text) <= limit else text[:limit] + "..."


def log_tool_calls(result: dict, request_id: str) -> None:
    """Observational post-run logging for a LangChain agent result."""
    for message in result["messages"]:
        if not isinstance(message, AIMessage) or not message.tool_calls:
            continue

        for tool_call in message.tool_calls:
            logger.info(
                "agent_tool_call",
                extra={
                    "event": "tool_call",
                    "request_id": request_id,
                    "tool_name": tool_call["name"],
                    "args_summary": summarize_args(tool_call["args"]),
                    "status": "requested",
                },
            )


request_id = str(uuid.uuid4())
started = time.perf_counter()

result_logged = agent.invoke(
    {"messages": [{"role": "user", "content": query}]}
)

logger.info(
    "agent_run_completed",
    extra={
        "event": "agent_run_completed",
        "request_id": request_id,
        "latency_ms": round((time.perf_counter() - started) * 1000, 2),
        "status": "success",
    },
)

log_tool_calls(result_logged, request_id)

print(result_logged["messages"][-1].content)

## What to build for clients

For a client-facing LangChain application, build beyond the notebook:

- An API service with authentication and tenant isolation.
- Typed request/response models.
- A limited set of read-only tools at first.
- Authorization and validation inside each tool.
- Structured logs or OpenTelemetry-compatible traces.
- Rate limits, timeouts, retries, and cost controls.
- An evaluation set with real client questions.
- A documented failure mode for model, tool, and upstream-service failures.

Keep the agent loop observable, but keep security and business rules in the tool execution boundary.